# Zero-Shot Model Comparison - Chronos & Kronos

In [ ]:
import os
from pathlib import Path

REPO_NAME = "ba"
REPO_URL = "https://github.com/bp571/ba.git"

if not Path(REPO_NAME).exists():
    !git clone {REPO_URL}

os.chdir(REPO_NAME)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q torch transformers peft gluonts pandas numpy tqdm PyYAML

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

MODEL = 'kronos'  # 'chronos' or 'kronos'
SEEDS = [13, 42, 123, 456, 789]
BATCH_SIZE = 48  # for kronos

BASE_PARAMS = {
    'context_steps': 80,
    'forecast_steps': 12,
    'stride_steps': 12,
    'steps': 120
}

print(f"Model: {MODEL}")
print(f"Seeds: {SEEDS}")
print(f"Params: {BASE_PARAMS}")

## Chronos Zero-Shot

In [ ]:
if MODEL == 'chronos':
    import json
    import time
    from datetime import datetime
    import pandas as pd
    from tqdm import tqdm
    
    from data.factory import DataFactory
    from core.model_loader import load_chronos_predictor
    from core.reproducibility import set_all_seeds
    from experiments.runner import run_rolling_benchmark
    
    factory = DataFactory(config_path="config/assets.yaml")
    tickers = factory.get_tickers()
    
    print(f"Assets: {len(tickers)}")
    
    for seed in SEEDS:
        print(f"\n{'='*60}\nSeed: {seed}\n{'='*60}")
        
        set_all_seeds(seed=seed)
        predictor = load_chronos_predictor()
        
        results_dir = Path(f"01_model_comparison/results/chronos/seed_{seed}")
        results_dir.mkdir(exist_ok=True, parents=True)
        
        all_results = {}
        start_time = time.time()
        
        for ticker in tqdm(tickers, desc=f"Seed {seed}"):
            try:
                df = factory.load_or_download(ticker)
                if df.empty:
                    continue
                
                test_start = pd.Timestamp('2021-01-01', tz='UTC')
                if isinstance(df.index, pd.DatetimeIndex):
                    df = df[df.index >= test_start]
                elif 'datetime' in df.columns:
                    df['datetime'] = pd.to_datetime(df['datetime'])
                    df = df[df['datetime'] >= test_start]
                
                if df.empty:
                    continue
                
                n_total = len(df)
                c, f, s = BASE_PARAMS['context_steps'], BASE_PARAMS['forecast_steps'], BASE_PARAMS['stride_steps']
                max_steps = (n_total - c - f) // s + 1
                
                current_params = BASE_PARAMS.copy()
                current_params['steps'] = max(0, min(BASE_PARAMS['steps'], max_steps))
                
                if current_params['steps'] == 0:
                    continue
                
                result = run_rolling_benchmark(predictor, df, ticker, current_params)
                
                if result:
                    all_results[ticker] = result['metrics']
                    
                    output_path = results_dir / f"result_{ticker}.json"
                    with open(output_path, 'w') as f:
                        json.dump(result, f, indent=4)
                        
            except Exception as e:
                pass
        
        final_path = results_dir / "final_energy_study.json"
        with open(final_path, 'w') as f:
            json.dump({
                'timestamp': datetime.now().isoformat(),
                'model': 'Chronos',
                'random_seed': seed,
                'params': BASE_PARAMS,
                'processing_time_seconds': time.time() - start_time,
                'n_assets_processed': len(all_results),
                'summary': all_results
            }, f, indent=4)
        
        print(f"Completed in {time.time() - start_time:.1f}s - {len(all_results)} assets")

## Kronos Zero-Shot

In [ ]:
if MODEL == 'kronos':
    import json
    import time
    from datetime import datetime
    import pandas as pd
    from tqdm import tqdm
    
    from data.factory import DataFactory
    from core.model_loader import load_kronos_predictor
    from core.reproducibility import set_all_seeds
    from experiments.runner import run_rolling_benchmark_multi_asset
    
    factory = DataFactory(config_path="config/assets.yaml")
    tickers = factory.get_tickers()
    
    print(f"Assets: {len(tickers)}")
    
    for seed in SEEDS:
        print(f"\n{'='*60}\nSeed: {seed}\n{'='*60}")
        
        set_all_seeds(seed=seed)
        predictor = load_kronos_predictor()
        
        results_dir = Path(f"01_model_comparison/results/kronos/seed_{seed}")
        results_dir.mkdir(exist_ok=True, parents=True)
        
        asset_data = {}
        skipped = []
        
        for ticker in tqdm(tickers, desc="Loading assets"):
            try:
                df = factory.load_or_download(ticker)
                if df.empty:
                    skipped.append(ticker)
                    continue
                
                test_start = pd.Timestamp('2021-01-01', tz='UTC')
                if isinstance(df.index, pd.DatetimeIndex):
                    df = df[df.index >= test_start]
                elif 'datetime' in df.columns:
                    df['datetime'] = pd.to_datetime(df['datetime'])
                    df = df[df['datetime'] >= test_start]
                
                if df.empty:
                    skipped.append(ticker)
                    continue
                
                if 'datetime' not in df.columns:
                    df = df.reset_index().rename(columns={df.index.name: 'datetime', 'date': 'datetime'})
                
                n_total = len(df)
                c, f, s = BASE_PARAMS['context_steps'], BASE_PARAMS['forecast_steps'], BASE_PARAMS['stride_steps']
                max_steps = (n_total - c - f) // s + 1
                
                if max_steps <= 0:
                    skipped.append(ticker)
                    continue
                
                asset_data[ticker] = df
                
            except Exception as e:
                skipped.append(ticker)
        
        print(f"Loaded {len(asset_data)} assets, skipped {len(skipped)}")
        
        if not asset_data:
            continue
        
        start_time = time.time()
        
        all_results = run_rolling_benchmark_multi_asset(
            predictor=predictor,
            asset_data_dict=asset_data,
            params=BASE_PARAMS,
            batch_size=BATCH_SIZE,
            verbose=True
        )
        
        final_summary = {}
        for ticker, result in all_results.items():
            if result:
                final_summary[ticker] = result['metrics']
                
                output_path = results_dir / f"result_{ticker}.json"
                with open(output_path, 'w') as f:
                    json.dump(result, f, indent=4)
        
        final_path = results_dir / "final_energy_study.json"
        with open(final_path, 'w') as f:
            json.dump({
                'timestamp': datetime.now().isoformat(),
                'model': 'Kronos',
                'random_seed': seed,
                'params': BASE_PARAMS,
                'batch_size': BATCH_SIZE,
                'processing_time_seconds': time.time() - start_time,
                'n_assets_processed': len(all_results),
                'summary': final_summary
            }, f, indent=4)
        
        print(f"Completed in {time.time() - start_time:.1f}s - {len(all_results)} assets")

## Summary

In [ ]:
import json
import pandas as pd

results = []

for seed in SEEDS:
    final_path = Path(f"01_model_comparison/results/{MODEL}/seed_{seed}/final_energy_study.json")
    if final_path.exists():
        with open(final_path) as f:
            data = json.load(f)
        
        results.append({
            'seed': seed,
            'n_assets': data['n_assets_processed'],
            'time_s': data['processing_time_seconds']
        })

df = pd.DataFrame(results)
print("\n" + "="*60)
print(f"ZERO-SHOT EVALUATION - {MODEL.upper()}")
print("="*60)
print(df.to_string(index=False))
print(f"\nTotal time: {df['time_s'].sum():.1f}s")
print(f"Avg time per seed: {df['time_s'].mean():.1f}s")